# Preprocessing

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_DEFAULT_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "ibm-granite/granite-4.1-8b"

In [ ]:
from wget import download

from langchain_text_splitters import CharacterTextSplitter

from langchain_community.document_loaders import TextLoader

from langchain_openrouter import ChatOpenRouter

from langchain_openai import OpenAIEmbeddings

from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.messages import HumanMessage, AIMessage


In [ ]:
def llm_model(params=None):

    # 1. Define sensible defaults
    config = {
        "model": OPENROUTER_MODEL,
        "api_key": OPENROUTER_API_KEY,
        "base_url": OPENROUTER_DEFAULT_BASE_URL,
        "temperature": 0.5,
        "max_tokens": 256,
        "max_completion_tokens": 128
    }
    
    if params:
        config.update(params)
        
    # 3. Initialize the model
    model = ChatOpenRouter(
        model=config["model"],
        api_key=config["api_key"],
        base_url=config["base_url"],
        temperature=config["temperature"],
        max_tokens=config["max_tokens"],
        max_completion_tokens=config["max_completion_tokens"]
    )

    return model

def llm_model_response(prompt_text, params=None):
            
    # 3. Initialize the model
    model = llm_model(params)

    response = model.invoke(prompt_text)

    return response

## Load The Document

In [ ]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Use wget to download the file
if os.path.exists(filename):
    os.remove(filename)
download(url, out=filename)
print('file downloaded')

with open(filename, 'r') as file:
    # Read the contents of the file
    contents = file.read()
    print(contents[:1000])

## Splitting the document into chunks

In [ ]:
loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunks = text_splitter.split_documents(documents)
print(len(chunks))

## Embedding and storing


In [ ]:
embedding_model_name = "openai/text-embedding-3-small"
    
texts = [chunk.page_content for chunk in chunks]

openai_embeddings = OpenAIEmbeddings(model=embedding_model_name, openai_api_key=OPENROUTER_API_KEY, openai_api_base=OPENROUTER_DEFAULT_BASE_URL, dimensions=1024)
chroma_doc_search = Chroma.from_texts(texts, openai_embeddings, collection_name="company-policies")
print(chroma_doc_search.get())

# LLM Model Construction

## Model

In [ ]:
llm_model_name = "openai/gpt-4o-mini"

params = {
    "model": llm_model_name,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

## Integrating Langchain

In [ ]:
# already using prompt template, exercise was pushing for RetrievalQA, wich is deprecated

retriever = chroma_doc_search.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Use the given context to answer the question. If you don't know, say you don't know. Context: {context}"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(model, prompt)
qa = create_retrieval_chain(retriever, question_answer_chain)

query = "what is mobile policy?"
result = qa.invoke({"input": query})

print(result["answer"])

# Dive Deeper

## Prompt Template

Already created PromptTemplate previsously, exercise was pushing for RetrievalQA, wich is deprecated

In [ ]:
query = "Can I eat in company vehicles?"
result = qa.invoke({"input": query})

print(result["answer"])

## Make the conversation have memory

As the model does not have memory, it won't relate to it as the car
Sample response:

```Based on the Internet and Email Policy, you cannot:
1. Use company-provided internet and email services for non-job-related tasks during work hours, except for limited personal use during non-work hours that does not interfere with work responsibilities.
2. Share your login credentials or passwords with others.
3. Open email attachments or click on links from unknown sources without exercising caution.
4. Transmit confidential information, trade secrets, or sensitive customer data via email without using encryption.
5. Discuss company matters on public forums or social media without discretion.
6. Engage in harassment, discrimination, or distribute offensive or inappropriate content.
7. Violate relevant laws and regulations regarding internet and email usage, including copyright and data protection.
8. Use company internet and email in ways that may lead to disciplinary measures or termination for policy violations.
It is important to adhere to these guidelines to ensure responsible and secure usage of digital communication tools.
´´´

In [ ]:
query = "What I cannot do in it?"
result = qa.invoke({"input": query})

print(result["answer"])

In [ ]:
# Create a retriever from the Chroma vector store
# This object is responsible for searching the embedded document chunks
# and returning the most relevant ones for a given question
retriever = chroma_doc_search.as_retriever()


# ------------------------------------------------------------
# 1) Prompt used to rewrite follow-up questions into standalone
#    search queries before retrieval
# ------------------------------------------------------------
# Why this is needed:
# If the user asks a follow-up like "What can I not do in it?",
# the retriever alone does not know what "it" refers to.
# This prompt tells the LLM to convert that follow-up into a
# complete, self-contained question using chat history.
contextualize_q_system_prompt = (
    "Given the chat history and the latest user question, "
    "formulate a standalone question that can be understood "
    "without the chat history. Do NOT answer the question; "
    "just rewrite it if needed, otherwise return it as is."
)


# Build the prompt template for question rewriting
# - system: instructions for the model
# - MessagesPlaceholder("chat_history"): injects previous conversation
# - human: the latest user input
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])


# Wrap the original retriever with a history-aware retriever
# What this new retriever does:
# 1. Looks at chat_history + current question
# 2. Rewrites the current question into a standalone version if needed
# 3. Sends that rewritten question to the retriever
# This improves retrieval quality for conversational follow-ups
history_aware_retriever = create_history_aware_retriever(
    model,                   # LLM used to rewrite the question
    retriever,               # Base retriever that searches Chroma
    contextualize_q_prompt   # Prompt that controls rewriting behavior
)


# ------------------------------------------------------------
# 2) Prompt used to answer the question using retrieved context
#    and chat history
# ------------------------------------------------------------
# This prompt tells the model to answer only from retrieved content.
# If the answer is not in the retrieved documents, it should say
# it does not know instead of inventing an answer.
qa_system_prompt = (
    "Use the given retrieved context to answer the user's question. "
    "If you don't know, say you don't know.\n\n"
    "Context: {context}"
)


# Build the final QA prompt
# - system: tells the model how to answer
# - chat_history: includes previous conversation turns
# - human: includes the user's current question
# The retrieved documents will be inserted into {context}
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])


# Create a chain that takes retrieved documents and "stuffs" them
# into the QA prompt before calling the model
# In other words, this is the part that generates the final answer
question_answer_chain = create_stuff_documents_chain(model, qa_prompt)


# Create the full conversational retrieval chain
# This combines:
# - the history-aware retriever (find relevant documents)
# - the QA chain (answer using those documents)
#
# Flow:
# current question + chat history
# -> rewrite question if needed
# -> retrieve relevant document chunks
# -> generate final answer using retrieved context
qa_with_memory = create_retrieval_chain(history_aware_retriever, question_answer_chain)


# Initialize an in-memory list to store the conversation history
# This will hold HumanMessage and AIMessage objects during the notebook session
# Each time you ask a question and get an answer, you append both to this list
chat_history = []

In [ ]:
# Ask the first question about the company's mobile policy
query = "what is the mobile policy?"

# Run the retrieval + memory-enabled QA chain
# - "input" is the current user question
# - "chat_history" contains previous conversation messages
#   (it's empty on the first question)
result = qa_with_memory.invoke({
    "input": query,
    "chat_history": chat_history
})

# Print the model's answer to the question
print(result["answer"])

# Save this question/answer pair into chat history
# so future follow-up questions can use this context
chat_history.extend([
    HumanMessage(content=query),              # Store the user's question
    AIMessage(content=result["answer"])       # Store the assistant's answer
])

# Ask a second question about eating in company vehicles
query = "Can i eat in company vehicles?"

# Run the same chain again, now with updated chat history
# The model can use the previous exchange as conversational context
result = qa_with_memory.invoke({
    "input": query,
    "chat_history": chat_history
})

# Print the answer to the second question
print(result["answer"])

# Add the second question and answer to chat history
# so the conversation continues to build context over time
chat_history.extend([
    HumanMessage(content=query),              # Store second user question
    AIMessage(content=result["answer"])       # Store second assistant answer
])


# Ask a follow-up question that uses a vague reference: "it"
# Here, "it" likely refers to "company vehicles" from the previous question
query = "What I cannot do in it?"

# Because chat_history is included, the chain can interpret "it"
# using previous conversation context before retrieving documents
result = qa_with_memory.invoke({
    "input": query,
    "chat_history": chat_history
})

# Print the answer to the follow-up question
print(result["answer"])

# Store the third question and answer as well
# This keeps the conversation state complete for future turns
chat_history.extend([
    HumanMessage(content=query),              # Store third user question
    AIMessage(content=result["answer"])       # Store third assistant answer
])

## Wrap it and make it an agent

In [ ]:
def qa_agent ():
        
    # Create a retriever from the Chroma vector store
    # This object is responsible for searching the embedded document chunks
    # and returning the most relevant ones for a given question
    qa_agent_retriever = chroma_doc_search.as_retriever()
    
    # ------------------------------------------------------------
    # 1) Prompt used to rewrite follow-up questions into standalone
    #    search queries before retrieval
    #------------------------------------------------------------
    # Why this is needed:
    # If the user asks a follow-up like "What can I not do in it?",
    # the retriever alone does not know what "it" refers to.
    # This prompt tells the LLM to convert that follow-up into a
    # complete, self-contained question using chat history.
    qa_agent_contextualize_q_system_prompt = (
        "Given the chat history and the latest user question, "
        "formulate a standalone question that can be understood "
        "without the chat history. Do NOT answer the question; "
        "just rewrite it if needed, otherwise return it as is."
    )


    # Build the prompt template for question rewriting
    # - system: instructions for the model
    # - MessagesPlaceholder("chat_history"): injects previous conversation
    # - human: the latest user input
    qa_agent_contextualize_q_prompt = ChatPromptTemplate.from_messages([
        ("system", qa_agent_contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])


    # Wrap the original retriever with a history-aware retriever
    # What this new retriever does:
    # 1. Looks at chat_history + current question
    # 2. Rewrites the current question into a standalone version if needed
    # 3. Sends that rewritten question to the retriever
    # This improves retrieval quality for conversational follow-ups
    history_aware_retriever = create_history_aware_retriever(
        model,                   # LLM used to rewrite the question
        qa_agent_retriever,      # Base retriever that searches Chroma
        qa_agent_contextualize_q_prompt   # Prompt that controls rewriting behavior
    )


    # ------------------------------------------------------------
    # 2) Prompt used to answer the question using retrieved context
    #    and chat history
    # ------------------------------------------------------------
    # This prompt tells the model to answer only from retrieved content.
    # If the answer is not in the retrieved documents, it should say
    # it does not know instead of inventing an answer.
    qa_agent_system_prompt = (
        "Use the given retrieved context to answer the user's question. "
        "If you don't know, say you don't know.\n\n"
        "Context: {context}"
    )


    # Build the final QA prompt
    # - system: tells the model how to answer
    # - chat_history: includes previous conversation turns
    # - human: includes the user's current question
    # The retrieved documents will be inserted into {context}
    qa_agent_prompt = ChatPromptTemplate.from_messages([
        ("system", qa_agent_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])


    # Create a chain that takes retrieved documents and "stuffs" them
    # into the QA prompt before calling the model
    # In other words, this is the part that generates the final answer
    qa_agent_question_answer_chain = create_stuff_documents_chain(model, qa_agent_prompt)


    # Create the full conversational retrieval chain
    # This combines:
    # - the history-aware retriever (find relevant documents)
    # - the QA chain (answer using those documents)
    #
    # Flow:
    # current question + chat history
    # -> rewrite question if needed
    # -> retrieve relevant document chunks
    # -> generate final answer using retrieved context
    qa_agent_with_memory = create_retrieval_chain(history_aware_retriever, qa_agent_question_answer_chain)


    # Initialize an in-memory list to store the conversation history
    # This will hold HumanMessage and AIMessage objects during the notebook session
    # Each time you ask a question and get an answer, you append both to this list
    qa_agent_chat_history = []

    # Start an interactive question loop
    while True:
    
        qa_agent_query = input("Question: ")

        # Exit the chat loop if the user types a stop word
        if qa_agent_query.lower() in ["quit", "exit", "bye"]:
            print("Answer: Goodbye!")
            break

        # Invoke the conversational retrieval chain
        # - input: current user question
        # - chat_history: all previous turns
        qa_agent_result = qa_agent_with_memory.invoke({
            "input": qa_agent_query,
            "chat_history": chat_history
        })

        # Print the generated answer
        print("Answer:", qa_agent_result["answer"])

        # Save the new conversation turn into history
        # so future follow-up questions can use this context
        qa_agent_chat_history.extend([
            HumanMessage(content=qa_agent_query),
            AIMessage(content=qa_agent_result["answer"])
        ])

In [ ]:
qa_agent()  